In [1]:
#Data Gathering
import pandas as pd
train=pd.read_csv('IMDB_Train.csv')
test=pd.read_csv('IMDB_Test.csv')

In [2]:
#EDA
train.shape

(40000, 2)

In [3]:
test.shape

(10000, 2)

In [4]:
train.head()

,review,sentiment
0,That's what I kept asking myself during the ma...,negative
1,I did not watch the entire movie. I could not ...,negative
2,A touching love story reminiscent of In the M...,positive
3,This latter-day Fulci schlocker is a totally a...,negative
4,"First of all, I firmly believe that Norwegian ...",negative


In [5]:
test.head()

,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative


In [6]:
#Checking for null value in training data 
train.isna().sum()

review       0
sentiment    0
dtype: int64

In [7]:
#looking at multiple review to figure out the preprocessing
train['review'][0]

'That\'s what I kept asking myself during the many fights, screaming matches, swearing and general mayhem that permeate the 84 minutes. The comparisons also stand up when you think of the one-dimensional characters, who have so little depth that it is virtually impossible to care what happens to them. They are just badly written cyphers for the director to hang his multicultural beliefs on, a topic that has been done much better in other dramas both on TV and the cinema.<br /><br />I must confess, I\'m not really one for spotting bad performances during a film, but it must be said that Nichola Burley (as the heroine\'s slutty best friend) and Wasim Zakir (as the nasty, bullying brother) were absolutely terrible. I don\'t know what acting school they graduated from, but if I was them I\'d apply for a full refund post haste. Only Samina Awan in the lead role manages to impress in a cast of so-called British talent that we\'ll probably never hear from again. At least, that\'s the hope. Ne

In [8]:
train['review'][100]

'This film got terrible reviews but because it was offbeat and because critics don\'t usually "get" offbeat films, I thought I\'d give it a try. Unfortunately they were largely right in this instance.<br /><br />The film just has an awkward feel too it that is most off putting. The sort of feel that is impossible to describe, but it\'s not a good one. To further confound things, the script is a dull aimless thing that is only vaguely interesting.<br /><br />The immensely talented Thurman just drifts through this mess creating barely an impact. Hurt and Bracco try in vain to add something to the film with enthusiastic performance but there is nothing in the script. It may have been less embarrassing for them if they had merely chosen to drift and get it over with like Thurman.<br /><br />One thing the "esteemed" film critics did fail to mention however is that the film is actually quite funny. Whether it be moments of accurate satire or some outrageously weird moments like when the cowg

In [9]:
#Observation=We were able to find html tags non alpha characters as wel as we where able to see did'nt etc contraction
#needed

In [10]:
#Exploreing the sentiment column
train['sentiment'].value_counts()

sentiment
negative    20039
positive    19961
Name: count, dtype: int64

In [11]:
#Observation =We saw 78 review is imbalanced but 78 review is negligible so we will not do any process on it

In [12]:
#Checking for duplicates
train['review'].duplicated().sum()

np.int64(257)

In [13]:
#Observation=We where able to foud 257 duplicate records which can turn the model to wards bais(overfitting) slighty 
#We decide to drop them

In [14]:
train= train.drop_duplicates()
print(train.shape)
print(train.duplicated().sum())

(39743, 2)
0


In [15]:
#We will check for test also
test['sentiment'].value_counts()

sentiment
positive    5039
negative    4961
Name: count, dtype: int64

In [16]:
#Checking for duplicates
test['review'].duplicated().sum()

np.int64(22)

In [17]:
test = test.drop_duplicates().reset_index(drop=True)

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

Train Shape: (39743, 2)
Test Shape : (9978, 2)


In [18]:
#NOW we got a good idea for inside now we go further for preprocessing

In [19]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [20]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [21]:
stop_words=set(stopwords.words("english"))-{'not','no','nor'}
lemmatizer=WordNetLemmatizer()

In [22]:
def preprocess(review):
    # Convert to lowercase
    review = review.lower()

    # Remove HTML tags
    review=re.sub(r"<.*?>",' ',review)
    # Remove URLs
    review = re.sub(r"http\S+|www\S+", "", review)

    # Keep only alphabets and spaces
    review = re.sub(r"[^a-z\s]", " ", review)

    # Remove extra spaces
    review = re.sub(r"\s+", " ", review).strip()

    # Tokenize
    words = review.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        
    ]

    return " ".join(words)

In [23]:
train["review"] = train["review"].apply(preprocess)
test["review"] = test["review"].apply(preprocess)

In [24]:
#Check
train['review'].head()

0    that s what i kept asking myself during the ma...
1    i did not watch the entire movie i could not w...
2    a touching love story reminiscent of in the mo...
3    this latter day fulci schlocker is a totally a...
4    first of all i firmly believe that norwegian m...
Name: review, dtype: object

In [25]:
#Now Converting the target column
train["sentiment"] = train["sentiment"].map({
    "negative": 0,
    "positive": 1
})

test["sentiment"] = test["sentiment"].map({
    "negative": 0,
    "positive": 1
})

In [26]:
print(train["sentiment"].value_counts())
print(test["sentiment"].value_counts())

sentiment
1    19887
0    19856
Name: count, dtype: int64
sentiment
1    5035
0    4943
Name: count, dtype: int64


In [27]:
#Tokenization

In [28]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [29]:
tokenizer=Tokenizer(num_words=80000, oov_token="<OOV>") #Top 50k words to get re=id of rare words 

In [30]:
#tokenizer.fit_on_texts(train['review'])

In [31]:
tokenizer.fit_on_texts(train['review'])

In [32]:
len(tokenizer.word_index)

81846

In [33]:
x_train = tokenizer.texts_to_sequences(train['review'])
x_test = tokenizer.texts_to_sequences(test['review'])

#### Padding

In [34]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [35]:
length = []
for i in x_train:
  length.append(len(i))

In [36]:
max(length)

2494

In [37]:
min(length)

6

In [38]:
sum(length) / len(length)

234.1318722793951

In [39]:
import numpy as np

review_lengths = [len(review) for review in x_train]

print("Minimum:", np.min(review_lengths))
print("Average:", np.mean(review_lengths))
print("Maximum:", np.max(review_lengths))
print("90th percentile:", np.percentile(review_lengths, 90))
print("95th percentile:", np.percentile(review_lengths, 95))
print("99th percentile:", np.percentile(review_lengths, 99))

Minimum: 6
Average: 234.1318722793951
Maximum: 2494
90th percentile: 456.8000000000029
95th percentile: 598.0
99th percentile: 915.0


In [40]:
# Now we do padding 

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_train_padded=pad_sequences(x_train,maxlen=598)
x_test_padded=pad_sequences(x_test,maxlen=598)

In [42]:
y_train=train['sentiment']
y_test=test['sentiment']

In [43]:
#Model Building

In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Dropout,SpatialDropout1D,Bidirectional
from tensorflow.keras.regularizers import l2

In [45]:
model=Sequential()
model.add(Embedding(input_dim=80000+1,
output_dim=200,
input_length=598))
model.add(SpatialDropout1D(0.2))
model.add(Bidirectional(LSTM(200,dropout=0.2,recurrent_dropout=0.2,return_sequences=True,kernel_regularizer=l2)))
model.add(Bidirectional(LSTM(64,dropout=0.3
        )
    )
)
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))

d:\Projects\IMBD\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [46]:
from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.0005)
model.compile(loss='binary_crossentropy',
optimizer=optimizer,metrics=['accuracy'])

In [47]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [48]:
model.fit(x_train_padded,
y_train,
validation_split=0.2,
epochs=30,
callbacks=[early_stop],
batch_size=128,
)

Epoch 1/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 627s 2s/step - accuracy: 0.7464 - loss: 1.6962 - val_accuracy: 0.8263 - val_loss: 0.4463
Epoch 2/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 618s 2s/step - accuracy: 0.8995 - loss: 0.2860 - val_accuracy: 0.8666 - val_loss: 0.3398
Epoch 3/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 616s 2s/step - accuracy: 0.9349 - loss: 0.1996 - val_accuracy: 0.8843 - val_loss: 0.3159
Epoch 4/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 617s 2s/step - accuracy: 0.9507 - loss: 0.1591 - val_accuracy: 0.8762 - val_loss: 0.3516
Epoch 5/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 614s 2s/step - accuracy: 0.9621 - loss: 0.1261 - val_accuracy: 0.8571 - val_loss: 0.5235
Epoch 6/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 614s 2s/step - accuracy: 0.9749 - loss: 0.0924 - val_accuracy: 0.8744 - val_loss: 0.3924
Epoch 7/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 640s 3s/step - accuracy: 0.9778 - loss: 0.0867 - val_accuracy: 0.8709 - val_loss: 0.4353
Epoch 8/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 701s 3s/step - accuracy: 0.9818 - loss: 0.0716 - val_accu

In [49]:
prob = model.predict(x_train_padded)

y_pred = (prob > 0.5).astype(int)

y_act = y_train

1242/1242 ━━━━━━━━━━━━━━━━━━━━ 265s 213ms/step


In [50]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_act, y_pred)

In [51]:
report = classification_report(y_act, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.94      0.96      0.95     19856
           1       0.96      0.94      0.95     19887

    accuracy                           0.95     39743
   macro avg       0.95      0.95      0.95     39743
weighted avg       0.95      0.95      0.95     39743



In [52]:
prob = model.predict(x_test_padded)

y_pred = (prob > 0.5).astype(int)

y_act = y_test

312/312 ━━━━━━━━━━━━━━━━━━━━ 66s 210ms/step


In [53]:
cm = confusion_matrix(y_act, y_pred)

In [54]:
report = classification_report(y_act, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.87      0.90      0.89      4943
           1       0.90      0.87      0.89      5035

    accuracy                           0.89      9978
   macro avg       0.89      0.89      0.89      9978
weighted avg       0.89      0.89      0.89      9978

